# 통계기초, 알고리즘, 이미지 분류 모델링, 시계열 예측 모델링 코테 준비

In [3]:
# 통계 기초 [설비 온도와 불량률의 상관관계]

# 문제 1. [통계기초] 설비 온도와 불량률의 상관관계 
# (단순 회귀)상황: 당신은 공정 설비의 온도(X, 독립변수)가 올라갈수록 제품의 불량률(y, 종속변수)도 증가하는지 확인하려 합니다.
# 요구사항: 
# 1.  scikit-learn을 사용하여 단순 선형 회귀(Simple Linear Regression) 모델을 학습시키는 코드를 작성하세요.
# 2.  학습된 모델을 바탕으로 예측값을 뽑아낸 뒤, 모델의 성능을 평가하기 위해 **MSE(평균 제곱 오차)**와 **$R^2$ Score(결정계수)**를 계산하여 출력하는 코드를 구현하세요.


import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

def analyze_temperature_defect():
    # 1. 가상의 센서 데이터 생성(온도와 불량률)
    # 여기서 scikit-learn의 X(독립변수)는 무조건 2차원 배열임
    X_temp = np.array([20, 22, 25, 27, 30])
    X = X_temp.reshape(-1, 1) 
    
    y_defect = np.array([1.2, 1.5, 2.1, 2.5, 3.2])
    
    # 2. 모델 선언 및 학습
    model = LinearRegression()
    model.fit(X, y_defect)
    
    
    # 3. 학습된 모델로 예측값 뽑아보기
    y_pred = model.predict(X)
    
    # 4. 모델 성능 평가 (채점)
    mse = mean_squared_error(y_defect,y_pred)
    r2 = r2_score(y_defect, y_pred)
    
    print(f"MSE (평균 제곱 오차) : {mse:.4f}")
    print(f"R^2 Score (결정계수) : {r2:.4f}")
    
    return model

analyze_temperature_defect()

MSE (평균 제곱 오차) : 0.0024
R^2 Score (결정계수) : 0.9953


,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [4]:
# 문제 2. [이미지 분류] 과적합을 방지하는 결함 분류 모델링
# 상황: 현미경으로 촬영한 반도체 표면 이미지 데이터셋이 주어졌습니다. 정상(Normal)과 결함(Defect) 두 가지로 분류해야 합니다. 데이터 개수가 많지 않은 상황입니다.

# 요구사항:

# PyTorch를 사용하여 EfficientNet-Base (또는 본인이 가장 자신 있는 백본 모델)의 사전 학습된 가중치(Pre-trained weights)를 불러와 이진 분류용으로 전이 학습(Transfer Learning) 뼈대를 구성하세요.

# 데이터가 적어 발생할 수 있는 과적합(Overfitting)을 방지하기 위해, torchvision.transforms를 활용하여 이미지 좌우 반전, 약간의 회전 등을 포함하는 데이터 증강(Data Augmentation) 파이프라인 코드를 작성하세요.

import torch
import torch.nn as nn
from torchgen import model
import torchvision.transforms as transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights


def build_defect_classifier():
    # 1. 과적합 방지를 위한 데이터 증강 파이프라인 구성
    train_transforms = transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),  # 좌우 반전
        transforms.Resize((224, 224)),  # EfficientNet 입력 크기에 맞게 조정
        transforms.RandomRotation(degrees=15),  # 약간의 회전
        transforms.ToTensor(),  # 텐서로 변환
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # 사전 학습된 모델에 맞는 정규화
    ])
    # 2. EfficientNet-Base 모델 불러오기 (사전 학습된 가중치 포함)
    # weights 매개변수를 통해 ImageNet으로 사전 학습된 가중치를 불러옵니다.
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
    
    # 3. 마지막 출력 레이어(Classifier) 수정
    # EfficientNet-B0의 기본 출력 레이어는 1000개의 클래스를 분류하도록 설계되어 있습니다.
    # 기존: (1) : Linear(in_features=1280, out_features=1000, bias=True)
    in_features = model.classifier[1].in_features  # 기존 출력 레이어의 입력 특징 수
    
    # 클래스 개수 2개 (정상=0, 결함=1) 로 수정
    model.classifier[1] = nn.Linear(in_features, 2)  # 새로운 출력 레이어로 교체
    
    return train_transforms, model


# 출력 확인용
_, my_model = build_defect_classifier()
print(my_model.classifier)


Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=2, bias=True)
)


In [5]:
# 문제 3. [시계열 예측] 진동 센서 데이터 슬라이딩 윈도우 구성
# 상황: 장비에 부착된 센서에서 1분 단위로 진동 수치가 1차원 배열로 계속 들어오고 있습니다. 우리는 과거 10분의 데이터를 보고, 바로 다음 1분(11분 째)의 진동 수치를 예측하려 합니다.

# 요구사항: 1.  전체 1차원 시계열 데이터(예: 길이가 1000인 리스트)가 주어졌을 때, 모델 학습을 위해 Window Size = 10 (입력 데이터 X), Horizon = 1 (예측할 타겟 데이터 y) 형태의 쌍(Pair)으로 변환하는 전처리 함수를 작성하세요.
# 2.  이 데이터를 입력으로 받을 수 있는 아주 간단한 LSTM 계층을 PyTorch(또는 TensorFlow/Keras)로 선언해 보세요.

import torch
import torch.nn as nn
import numpy as np

# 1. 슬라이딩 윈도우 전처리 함수
def creat_sequences(data, window_size):
    xs, ys = [], []
    # 데이터의 처음부터 (전체길이 - 윈도우 크기) 직전까지만 반복
    # 그래야 마지막 윈도우가 끝에 도달했을 때 그 다음 1개(y) 를 무사히 가져올 수 있음
    for i in range(len(data) - window_size):
        x_window = data[i : i + window_size] # 10개 추출
        y_target = data[i + window_size] # 그 다음 1개 추출
        
        xs.append(x_window)
        ys.append(y_target)
    
    # 모델에 넣기 위해 Tensor 형태로 변환
    # unsqueeze(-1)을 써서 마지막 차원(Feature 차원)을 1로 만들어줌 (LSTM은 3D 입력이 필요하기 때문)
    return torch.tensor(xs, dtype=torch.float32).unsqueeze(-1), torch.tensor(ys, dtype=torch.float32).unsqueeze(-1)

# 2. 간단한 LSTM 모델 선언
class SimpleLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=1):
        super(SimpleLSTM, self).__init__()
        # batch_first=True로 설정하면 입력과 출력 텐서의 형태가 (batch, seq, feature) 가 됨
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1) # 최종 출력은 1개 (다음 진동 수치 예측)
    
    def forward(self, x):
        # x 형태: (batch, seq_len, input_size)
        lstm_out, _ = self.lstm(x) # lstm_out 형태: (batch, seq_len, hidden_size)
        # 마지막 시점의 LSTM 출력만 사용하여 예측
        last_time_step = lstm_out[:, -1, :] # (batch, hidden_size)
        pred = self.fc(last_time_step) # (batch, 1)
        return pred
    
# 예시 데이터 생성 및 전처리
raw_sensor_data = [10, 12, 11, 15, 14, 16, 18, 17, 19, 21, 22, 25, 24]
window_size = 10

X, y = creat_sequences(raw_sensor_data, window_size)
model = SimpleLSTM()

# 예측 테스트 
# X의 형태: (3, 10, 1) -> 배치 사이즈 3, 윈도우 10, 피처 1
predictions = model(X)
print("입력 X 형태:", X.shape)
print("정답 y 형태:", y.shape)
print("모델 예측값:", predictions.detach().numpy())


입력 X 형태: torch.Size([3, 10, 1])
정답 y 형태: torch.Size([3, 1])
모델 예측값: [[0.2372258 ]
 [0.24661732]
 [0.2592573 ]]


📌 문제

다음 데이터를 이용하여 선형 회귀 모델을 학습하고 성능을 평가하시오.

입력
```
X = [1,2,3,4,5,6]
y = [2,4,5,4,5,7]
```

✅ 요구사항<br>

- train/test split (80:20)

- MSE, RMSE, R² 계산

- 잔차 출력


In [6]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

X = np.array([1,2,3,4,5,6]).reshape(-1,1)
y = np.array([2,4,5,4,5,7])

# train/test split
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# 모델 학습
model = LinearRegression()
model.fit(X_train, y_train)

# 예측
y_pred = model.predict(X_test)

# 평가
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R^2 Score: {r2:.4f}")

# 잔차
residuals = y_test - y_pred
print("잔차:", residuals)

MSE: 0.4450
RMSE: 0.6671
R^2 Score: 0.5550
잔차: [-0.5  0.8]


3️⃣ [이미지 분류 - CNN]

📌 문제

- MNIST 데이터셋을 이용하여 숫자 이미지 분류 모델을 구현하시오.

✅ 요구사항<br>

- CNN 모델 구현

- Accuracy 출력

- Overfitting 방지 기법 포함

- Confusion Matrix 출력

In [7]:
import tensorflow as tf 
from tensorflow.keras import layers, models
import numpy as np 
from sklearn.metrics import confusion_matrix

# 데이터 로드
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

# 전처리
X_train = X_train / 255.0
X_test = X_test / 255.0

# 모델 구성
model = models.Sequential([
    layers.Reshape((28,28,1), input_shape=(28,28)),
    layers.Conv2D(32, (3,3), activation='relu'),
    layers.MaxPooling2D(),
    layers.Dropout(0.3),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')
])

# 컴파일
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy',metrics=['accuracy'])
# 평가
loss, acc = model.evaluate(X_test, y_test)
print("Accuarcy : ",acc)

# Confusion Matrix
y_pred = np.argmax(model.predict(X_test), axis=1)
cm = confusion_matrix(y_test, y_pred)
print(cm)

c:\Programmers\.venv\lib\site-packages\keras\src\layers\reshaping\reshape.py:38: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.0663 - loss: 2.3179
Accuarcy :  0.06629999727010727
313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
[[  4 251   6   1  22 152   1 514  29   0]
 [132  25  13   0 267  65  84 509  40   0]
 [ 69 387   4   0 107 154   6 290  14   1]
 [ 29 185  73   0 106 427  11 179   0   0]
 [ 21  90 103  21 184  31   2 461  59  10]
 [ 15 225  91   2  90 211   7 236  13   2]
 [ 17 124  19   1  60 110   6 583  36   2]
 [ 32  40 145   3 139 438   1 225   1   4]
 [ 32 105  12   2 155  51   1 613   3   0]
 [ 32  72  36  17 219 104   1 524   3   1]]


4️⃣ [시계열 + 이상치 처리]

📌 문제

다음 시계열 데이터에서 이상치를 제거하고, 다음 값을 예측하시오.

입력
```
data = [100, 105, 110, 500, 120, 125, 130]
```

✅ 요구사항<br>

- 이상치 제거 (Z-score)

- sliding window 생성

- 회귀 모델 적용

- RMSE 계산

- 다음 값 예측

In [8]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error



In [11]:
data = np.array([100, 105, 110, 500, 120, 125, 130])

# 이상치 제거(Z-score)
mean = np.mean(data)
std = np.std(data)

filtered_data = data[(data > mean - 2*std) & (data < mean + 2*std)]

# sliding window 생성
X, y = [], []
for i in range(len(filtered_data) - 1):
    X.append([filtered_data[i]])
    y.append(filtered_data[i+1])
X = np.array(X)
y = np.array(y)

# train/test split
split = int(len(X)*0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

# 모델 학습
model = LinearRegression()
model.fit(X_train,y_train)

# 예측
pred = model.predict(X_test)

# 평가
rmse = np.sqrt(mean_squared_error(y_test, pred))
print(f"RMSE : {rmse}")

# 다음값 예측
next_value = model.predict([[filtered_data[-1]]])
print("Next value", next_value)

RMSE : 1.7142857142857224
Next value [136.85714286]


# 베이스라인 코드

## 문제 1. [통계기초] 장비 가동 시간(X)에 따른 마모도(y) 예측

- 상황: 가동 시간 데이터 X와 마모도 y가 주어집니다. 단순 선형 회귀 모델을 학습하고 $R^2$ 점수를 출력하세요.

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# 데이터 전처리
# 모델 선언 -> 학습 -> 예측 -> 평가 순서


def solutions(): 
    X_hours = [10,20,30,40,50]
    y_wear = [2.1, 3.8, 6.2, 8.0, 9.5]
    # X 데이터는 반드시 2차원으로
    X_reshaped = np.array(X_hours).reshape(-1,1)
    
    # 모델 선언 -> 학습(fit) -> 예측
    model = LinearRegression()
    model.fit(X_reshaped,y_wear)
    pred = model.predict(X_reshaped)
    
    # 평가
    score = r2_score(y_wear, pred) #실제값, 예측값 순서
    print(f"R2 score: {score}")
    
solutions()

R2 score: 0.994271234989534


## 문제 2. [이미지 분류] 불량 검출을 위한 EfficientNet-Base 전이 학습

- 상황: 3가지 상태(정상, 스크래치, 찍힘)를 분류해야 합니다. PyTorch를 이용해 모델 뼈대를 만드세요.

In [14]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

def build_vision_model():
    # EfficientNet-Base 모델 불러오기(Pre_trained 가중치 포함)
    model = efficientnet_b0(weights= EfficientNet_B0_Weights.DEFAULT)
    # efficientnet의 마지막 레이어 이름은 classifier 의 1번 인덱스임
    in_features = model.classifier[1].in_features
    # 우리가 원하는 클래스 갯수로 마지막층을 갈아끼움
    model.classifier[1] = nn.Linear(in_features, 3)
    
    return model

my_model = build_vision_model()
print("모델이 준비되었습니다! 마지막 층:", my_model.classifier[1])

모델이 준비되었습니다! 마지막 층: Linear(in_features=1280, out_features=3, bias=True)


# 문제 3. [시계열] 진동 데이터 슬라이딩 윈도우 생성

- 상황: 1차원 진동 데이터 리스트가 주어집니다. 과거 3개의 데이터(Window=3)를 보고 다음 1개를 예측할 수 있도록 X와 y 리스트를 만드세요.

In [15]:
def make_time_series_data(data, window_size):
    X,y = [], []
    
    # 인덱스 에러를 막기위한 마법의 반복문 조건
    for i in range(len(data) - window_size):
        # i부터 window_size 만큼 슬라이싱
        window = data[i : i + window_size]
        # 윈도우 바로 다음 데이터가 타겟(정답)
        target = data[i + window_size]
        
        X.append(window)
        y.append(target)
        
    return X, y

sensor_data = [1,2,3,4,5,6]
X_data, y_data = make_time_series_data(sensor_data, window_size=3)
print(f"X (입력): {X_data}")
print(f"y (정답): {y_data}")

X (입력): [[1, 2, 3], [2, 3, 4], [3, 4, 5]]
y (정답): [4, 5, 6]


# 문제 4. [알고리즘] 불량 부품 식별 (빈도수 세기)

- 상황: 검사 라인에서 부품 ID 배열이 주어집니다. 정확히 2번 불량이 발생한 부품의 ID만 모아서 리스트로 반환하세요.

In [18]:
def find_defective_parts(parts):
    counts = {}
    
    # 리스트 요소 개수 셀 때는 무조건 딕셔너리와 .get() 사용
    for part in parts:
        counts[part] = counts.get(part, 0) + 1
    result = []
    
    # 딕셔너리의 키와 값을 동시에 순회하는 .items()
    for part_id, count in counts.items():
        if count == 2:
            result.append(part_id)
    
    return result

parts_list = ["A", "B", "A", "C", "B", "B", "D", "A", "E", "E"]
print(find_defective_parts(parts_list))

['E']


# 문제 1. [데이터 전처리 / 알고리즘] 설비 센서 결측치 및 이상치 정제

- 상황: 장비의 온도 센서가 1분 단위로 데이터를 보냅니다. 통신 에러로 값이 누락(None)되거나, 센서 오작동으로 말도 안 되는 음수 값(이상치)이 섞여 들어옵니다.<br>
- 요구사항: 
    1. 온도가 0 미만인 값(이상치)은 직전의 정상 값으로 덮어씌우세요.
    2. None 값(결측치) 역시 직전의 정상 값으로 채워 넣으세요(Forward Fill).
    3. 첫 번째 데이터는 항상 정상이라고 가정합니다.

In [ ]:
def clean_sensor_data(data):
    cleaned = data.copy() # 원본데이터 훼손 방지
    
    for i in range(1, len(cleaned)):
        # 현재 값이 None이거나 0미만인 '비정상' 상태라면
        if cleaned[i] is None or cleaned[i] < 0:
            # 바로 직전값(이미 정제된 정상 값) 으로 덮어씌움
            cleaned[i] = cleaned[i-1]
            
    return cleaned

raw_data = [25.0, 25.5, None, 26.0, -50.0, 26.5, None]
print("정제된 데이터 : ", clean_sensor_data(raw_data))

정제된 데이터 :  [25.0, 25.5, 25.5, 26.0, 26.0, 26.5, 26.5]


# 문제 2. [통계/다중 회귀] 3가지 센서값을 이용한 장비 잔여 수명(RUL) 예측

- 상황: 이전에는 '온도' 하나만 봤지만, 실제로는 온도, 진동, 압력 3가지 데이터를 동시에 보고 장비가 며칠 뒤에 고장 날지(잔여 수명, RUL) 예측해야 합니다.

- 요구사항: scikit-learn을 사용하여 3개의 독립변수(Feature)를 가진 다중 선형 회귀 모델을 학습하고, 예측 오차인 **RMSE(평균 제곱근 오차)**를 구하세요.

In [20]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

def predict_remaining_useful_life():
    # X : [온도, 진동, 압력] 형태의 2차원 배열
    X = np.array([
        [45, 0.5, 1.2],
        [50, 0.6, 1.3],
        [60, 0.8, 1.5],
        [75, 1.2, 1.8],
        [80, 1.5, 2.0]
    ])
    # y는 고장까지 잔여 수명
    y = np.array([100, 85, 60, 20, 5])
    
    model = LinearRegression()
    model.fit(X, y)
    predictions = model.predict(X)
    
    # RMSE 는 MSE 의 루트
    # 오차의 단위를 직관적으로 보기위해 RMSE 사용
    mse = mean_squared_error(y, predictions)
    rmse = np.sqrt(mse)
    
    
    print(f"모델 RMSE 오차: {rmse:.2f}일")
    return model

predict_remaining_useful_life()

모델 RMSE 오차: 0.60일


,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


# 문제 3. [비전/알고리즘] 불량 영역 검출 정확도(IoU) 계산

- 상황: 비전 AI가 웨이퍼 표면의 스크래치 위치를 네모 박스(Bounding Box)로 찾았습니다. 사람이 직접 체크한 정답 박스와 AI가 찾은 박스가 얼마나 겹치는지를 나타내는 IoU(Intersection over Union) 지표를 계산해야 합니다.

- 요구사항: 두 박스 [x1, y1, x2, y2]가 주어질 때 (x1,y1은 좌상단, x2,y2는 우하단 좌표), 겹치는 영역의 넓이를 두 박스 전체의 넓이로 나눈 비율을 반환하세요.

In [21]:
def calculate_iou(box1, box2):
    # 겹치는 영역의 좌표 구하기
    inter_x1 = max(box1[0], box2[0])
    inter_y1 = max(box1[1], box2[1])
    inter_x2 = max(box1[2], box2[2])
    inter_y2 = max(box1[3], box2[3])
    
    # 겹치는 영역의 가로, 세로 길이(겹치지 않으면 0이 되도록 처리)
    inter_width = max(0, inter_x2 - inter_x1)
    inter_height = max(0, inter_y2 - inter_y1)
    intersection_area = inter_width * inter_height
    
    # 각 박스의 넓이 계산
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    # 합집합 넓이 = 박스1 넓이 + 박스2 넓이 - 교집합 넓이
    union_area = box1_area + box2_area - intersection_area
    
    # 분모가 0이 되는 경우 예외 처리(두 박스 넓이가 모두 0)
    if union_area == 0:
        return 0.0
    
    return intersection_area / union_area


answer_box = [10,20,50,50]
ai_box = [20,20,60,60]

print(f"불량 검출 IoU: {calculate_iou(answer_box, ai_box):.4f}")

불량 검출 IoU: 1.3333


# 문제 4. [시계열 심화] 다변량(Multivariate) 시계열 윈도우 생성

- 상황: 1회차에서는 1차원 리스트를 잘랐지만, 실제 데이터는 [온도, 진동]처럼 여러 피처(Feature)가 한 묶음으로 들어옵니다.

- 요구사항: 2차원 배열 데이터를 입력받아 Window Size만큼 자른 3차원 형태의 시계열 입력 데이터를 만드세요.

In [ ]:
import numpy as np

def make_multivariate_ts_data(data, window_size):
    X = []
    y = []
    
    # data가 [[온도1, 진동1], [온도2, 진동2], ...] 형태의 2차원 리스트
    for i in range(len(data) - window_size):
        window = data[i : i + window_size]
        target = data[i + window_size] # 타겟은 그 다음 시점의 데이터 묶음 전체를 예측 한다고 가정
        
        X.append(window)
        y.append(target)
        
    return np.array(X), np.array(y)

# [온도, 진동] 데이터 5개
multi_data = [[20, 0.1], [21, 0.2], [22, 0.3], [23, 0.4], [24, 0.5]]
X_multi, y_multi = make_multivariate_ts_data(multi_data, window_size=2)

print("X 형태 (Batch, Window, Feature):", X_multi.shape)
# 3개의 세트, 과거 2개 시점, 2개의 특징(온도/진동)

X 형태 (Batch, Window, Feature): (3, 2, 2)


# 심화버전